<a href="https://colab.research.google.com/github/HosseinMasoudi/Sentiment_Analysis_with_Transformers/blob/master/SentimentAnalysisTransformersNLP(WithOutHazm)(Use_Model_in_MLP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [2]:
!pip install num2fawords
!pip install parsivar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 60.0 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import tensorflow as tf
import transformers
import JackageNormalizer

from parsivar import Tokenizer
from JackageNormalizer import normalize_persian_text

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

from tensorflow.keras import layers, Model ,Input

from transformers import pipeline
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import DistilBertTokenizerFast
from transformers import TFDistilBertForSequenceClassification
from transformers import TFAutoModel

import tqdm as notebook_tqdm

In [4]:
model_name = "HooshvareLab/bert-fa-base-uncased"
num_labels = 2  #positive and negative

testing the pip line...

the model I chose only has a PyTorch checkpoint.

In [ ]:
# Use a pipeline as a high-level helper
pipe = pipeline("text-classification", model="HooshvareLab/bert-fa-base-uncased-sentiment-snappfood", framework="tf")
# Example usage of the pipeline
result = pipe("این غذا خیلی خوشمزه بود!")
print(result)

### 1. Load and Inspect Your Dataset

In [ ]:
df = pd.read_csv("/content/cleaned_snappfood copy.csv")
df.head()

,comment,label,comment_length,comment_cleaned
0,واقعا حیف وقت که بنویسم سرویس دهیتون شده افتضاح,0,47,واقعا حیف وقت که بنویسم سرویس دهیتون شده افتضاح
1,قرار بود ۱ ساعته برسه ولی نیم ساعت زودتر از مو...,1,132,قرار بود ساعته برسه ولی نیم ساعت زودتر از موق...
2,قیمت این مدل اصلا با کیفیتش سازگاری نداره، فقط...,0,89,قیمت این مدل اصلا با کیفیتش سازگاری نداره فقط ...
3,عالی بود همه چه درست و به اندازه و کیفیت خوب، ...,1,99,عالی بود همه چه درست و به اندازه و کیفیت خوب ا...
4,شیرینی وانیلی فقط یک مدل بود.,1,29,شیرینی وانیلی فقط یک مدل بود


In [ ]:
df.info()
df.isnull().count()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65973 entries, 0 to 65972
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   comment          65973 non-null  object
 1   label            65973 non-null  int64 
 2   comment_length   65973 non-null  int64 
 3   comment_cleaned  65973 non-null  object
dtypes: int64(2), object(2)
memory usage: 2.0+ MB


,0
comment,65973
label,65973
comment_length,65973
comment_cleaned,65973


In [ ]:
print(df.shape)
print(df.columns)

(65973, 4)
Index(['comment', 'label', 'comment_length', 'comment_cleaned'], dtype='object')


In [ ]:
df = df[['comment', 'label']]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
tokenizer = Tokenizer()

def clean_text(text):
    text = normalize_persian_text(text)
    tokens = tokenizer.tokenize_words(text)
    return ' '.join(tokens)


In [ ]:
df['comment'] = df['comment'].apply(clean_text)

df['comment'].sample(20)

,comment
10265,لازانیا اصلاکیفیت نداشت سس پنیر
37192,دایی خیلی دیر رسید
22779,اصلا طعمی نداشت ومزه نداشت بو میداد
54544,ممنون فست فود شیلا زود فود
33234,پای سیب سفارش واقعا تازه عالی بود بینهایت سپاس...
27340,اصلا سفارش نان ارسال نشده
5276,مشتری ثابت پیتزا سیب هستم واقعا غذایی دریافت ا...
46466,قیمت ظرف رو عدد دو هزار توان گرفتن یه ظرف بیکی...
58379,لطفا شکر آبمیوهها اضافه نکنید
40864,مواد غذایی مونده کهنه بسته بندی قشنگ


#### Split into train / val / test

In [ ]:
texts = df['comment'].tolist()
labels = df['label'].values

In [ ]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.3, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

70% → train

15% → validation

15% → test

In [ ]:
print(f"Train shape: {len(train_texts)}, Validation shape: {len(val_texts)}, Test shape: {len(test_texts)}\n")

print("Data loaded and processed successfully.")
print(f"Sample of cleaned text:\n ({train_texts[0]}) \n label: {train_labels[0]}")

Train shape: 46181, Validation shape: 9896, Test shape: 9896

Data loaded and processed successfully.
Sample of cleaned text:
 (سیبزمینی روغن مونده درست بو میداد قابل خوردن شکل شمایلش خوب نبود) 
 label: 0


### (Model Name and parameters)

Since the output of the SequenceClassification model was fixed and cannot be used in a feed-forward network, we must use the original parsBert model itself.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-fa-base-uncased-sentiment-snappfood")

In [ ]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    "HooshvareLab/bert-fa-base-uncased-sentiment-snappfood",
    num_labels=2,
    use_safetensors=True)

tf_model.h5:   0%|          | 0.00/652M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All model checkpoint layers were used when initializing TFBertForSequenceClassification.

All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at HooshvareLab/bert-fa-base-uncased-sentiment-snappfood.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


### 2. Prepare Text and Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def encode_texts(texts):
    return tokenizer(list(texts), truncation=True, padding=True, max_length=128)

In [ ]:
train_encoded = encode_texts(train_texts)
val_encoded = encode_texts(val_texts)
test_encoded = encode_texts(test_texts)

convert Encoding to TF DataSet

In [ ]:
def make_tf_dataset(encoding, labels, shuffle=False, BATCH_SIZE=16):
    dataset = tf.data.Dataset.from_tensor_slices((dict(encoding), labels))
    if shuffle:
        dataset = dataset.shuffle(1000)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
train_dataset = make_tf_dataset(train_encoded, train_labels, shuffle=True)
val_dataset = make_tf_dataset(val_encoded, val_labels)
test_dataset = make_tf_dataset(test_encoded, test_labels)

### 3. Laod the Model_Compile the Model_Train the Model

1. Inputs (all from tf.keras)
2. Transformer backbone (freeze the backbone completely)
3. Extract sentence representation (choose one option)
4. Head (MLP classifier)

In [6]:
def build_sentiment_model(model_name: str, num_labels: int):

    input_ids      = Input(shape=(None,), dtype=tf.int32, name="input_ids")
    attention_mask = Input(shape=(None,), dtype=tf.int32, name="attention_mask")

    transformer = TFAutoModel.from_pretrained(model_name)
    transformer.trainable = False

    outputs = transformer(input_ids=input_ids, attention_mask=attention_mask)
    cls = outputs.pooler_output

    x = layers.Dense(256, activation="relu", name="Dense")(cls)
    x = layers.Dropout(0.3, name="Dropout")(x)
    logits = layers.Dense(num_labels, activation="softmax", name="OutputSoftmax")(x)

    model = Model(inputs=[input_ids, attention_mask],outputs=logits)
    return model

model = build_sentiment_model(model_name, num_labels=2)

model.summary()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/440 [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/963M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at HooshvareLab/bert-fa-base-uncased were not used when initializing TFBertModel: ['nsp___cls', 'mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at HooshvareLab/bert-fa-base-uncased.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for pred

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, None)]               0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, None)]               0         []                            
 )                                                                                                
                                                                                                  
 tf_bert_model (TFBertModel  TFBaseModelOutputWithPooli   1628413   ['input_ids[0][0]',           
 )                           ngAndCrossAttentions(last_   44         'attention_mask[0][0]']      
                             hidden_state=(None, None,                                        

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-3, clipnorm=1.0),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_model_stage1.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),
]

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
# Accuracy
plt.plot(history.history['sparse_categorical_accuracy'], label='Training Accuracy')
plt.plot(history.history['val_sparse_categorical_accuracy'], label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

### 4. Evaluate on TEST set

In [ ]:
print("Final Evaluation on Test Set")
model.evaluate(test_dataset)

✅ Final Evaluation on Test Set
619/619 [==============================] - 57s 87ms/step - loss: 0.3776 - sparse_categorical_accuracy: 0.8516


[0.3776339590549469, 0.8515561819076538]

### 5. Save the Fine-Tuned Model

saving on GoogleDrive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


model.save_pretrained('/content/drive/MyDrive/mySentimentAnalysis_model')
tokenizer.save_pretrained('/content/drive/MyDrive/mySentimentAnalysis_model')